# P06 CoT and PRM Data Factory 源码全流程 Notebook

这本 notebook 不是通过 `subprocess` 调壳执行脚本，而是把 `project_6_prm_data` 里的完整流程源码直接展开到 notebook 中。

你可以把它当成一本按项目顺序组织的代码讲义来阅读：

- 主流程脚本按推荐执行顺序排在前面
- 辅助脚本和工具脚本排在后面
- 如果 README 里提到某个脚本但仓库里缺失，也会保留缺口说明

## 项目环境

- 项目目录：`project_6_prm_data`
- 建议 Conda 环境：`p6-prm`

## 本 notebook 收录的源码文件顺序

1. `src/sampler.py` - 采样任务与规格 [存在]
2. `src/generate_traces.py` - 生成推理轨迹 [存在]
3. `src/validate_and_score.py` - 校验并打分 [存在]
4. `src/prepare_prm_data.py` - 封装 PRM 数据 [存在]
5. `src/evaluate_prm.py` - 评估 PRM 数据集 [存在]
6. `src/run_p6_checks.py` - 项目检查 [存在]
7. `src/pipeline_utils.py` - 辅助脚本 [存在]

## 关键产物

- `data/processed/seed_pool.jsonl`
- `data/processed/task_spec.json`
- `data/processed/cot_traces.jsonl`
- `data/processed/trace_summary.json`
- `data/processed/validated_traces.jsonl`
- `data/processed/step_rewards.jsonl`
- `data/processed/validation_summary.json`
- `data/training/prm_step_dataset.jsonl`
- `data/training/train.jsonl`
- `data/training/val.jsonl`
- `data/training/smoke_test.jsonl`
- `data/training/training_manifest.json`
- `data/reports/p6_report.md`
- `data/reports/p6_metrics.json`
- `data/reports/p6_test_results.json`
- `data/reports/p6_test_report.md`


## 项目 README

下面直接附上项目自带的 `README.md`，方便在 notebook 里连同源码一起看。

# P6 CoT and PRM Data Factory

This project builds a small, reproducible CoT reasoning dataset and PRM training dataset.

## Scope

The current implementation covers:

1. Project goals and task setting
   - Math and code reasoning tasks.
   - Final-answer validation and step-level process supervision.
2. Reasoning trace generation
   - CoT, scratchpad-style steps, positive/negative/repair traces.
   - Step labels for correct, incorrect, and repaired reasoning.
3. Automatic validation and scoring
   - Rule checks for math answers.
   - Execution and unit tests for code traces.
   - Reward bucket assignment.
4. PRM data organization
   - Step-level JSONL slices.
   - Train/val/smoke splits.
5. Experimental results and review
   - Outcome-only versus process supervision comparison.
   - Quality issues and reward-bucket diagnostics.
6. Extension directions
   - Can extend to science reasoning, table reasoning, and agent planning.

## Environment

Dedicated conda environment:

```bash
conda activate p6-prm
```

Environment files:

- `environment.yml`
- `environment.lock.yml`

Recommended creation commands:

```bash
conda env create -f environment.yml
conda env update -n p6-prm -f environment.lock.yml --prune
```

## Recommended Run Order

```bash
python src/sampler.py
python src/generate_traces.py
python src/validate_and_score.py
python src/prepare_prm_data.py
python src/evaluate_prm.py
python src/run_p6_checks.py
```

## Main Outputs

- `data/processed/seed_pool.jsonl`
- `data/processed/task_spec.json`
- `data/processed/cot_traces.jsonl`
- `data/processed/trace_summary.json`
- `data/processed/validated_traces.jsonl`
- `data/processed/step_rewards.jsonl`
- `data/processed/validation_summary.json`
- `data/training/prm_step_dataset.jsonl`
- `data/training/train.jsonl`
- `data/training/val.jsonl`
- `data/training/smoke_test.jsonl`
- `data/training/training_manifest.json`
- `data/reports/p6_report.md`
- `data/reports/p6_metrics.json`
- `data/reports/p6_test_results.json`
- `data/reports/p6_test_report.md`


## 源码目录概览

当前 `src/` 中实际存在的 Python 文件共 `7` 个：

- `src/evaluate_prm.py`
- `src/generate_traces.py`
- `src/pipeline_utils.py`
- `src/prepare_prm_data.py`
- `src/run_p6_checks.py`
- `src/sampler.py`
- `src/validate_and_score.py`


## 完整源码展开


### `src/sampler.py` - 采样任务与规格

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter

from pipeline_utils import (
    GSM8K_FILE,
    MBPP_FILE,
    PROCESSED_DIR,
    ensure_standard_dirs,
    extract_final_answer,
    load_jsonl,
    split_reasoning_steps,
    write_json,
    write_jsonl,
)

SEED_FILE = PROCESSED_DIR / "seed_pool.jsonl"
TASK_SPEC_FILE = PROCESSED_DIR / "task_spec.json"

NUM_MATH = 24
NUM_CODE = 12


def infer_math_topic(question: str) -> str:
    text = question.lower()
    if "%" in text or "percent" in text or "tax" in text:
        return "percentages"
    if "mile" in text or "hour" in text or "minute" in text or "speed" in text:
        return "rates_and_motion"
    if "fraction" in text or "/" in text:
        return "fractions"
    return "arithmetic"


def infer_code_topic(prompt: str) -> str:
    text = prompt.lower()
    if "string" in text or "character" in text:
        return "string_manipulation"
    if "list" in text or "array" in text:
        return "list_processing"
    if "search" in text or "sort" in text:
        return "search_sort"
    return "function_design"


def main() -> None:
    ensure_standard_dirs()
    gsm8k = load_jsonl(GSM8K_FILE)
    mbpp = load_jsonl(MBPP_FILE)

    seeds: list[dict] = []
    for index, record in enumerate(gsm8k):
        final_answer = extract_final_answer(record["answer"])
        steps = split_reasoning_steps(record["answer"])
        if not final_answer or len(steps) < 2:
            continue
        seeds.append(
            {
                "seed_id": f"math_{index}",
                "domain": "math",
                "topic": infer_math_topic(record["question"]),
                "question": record["question"],
                "reference_steps": steps,
                "final_answer": final_answer,
                "source_dataset": "gsm8k_train",
            }
        )
        if sum(seed["domain"] == "math" for seed in seeds) >= NUM_MATH:
            break

    code_count = 0
    for record in mbpp:
        if len(record.get("test_list", [])) < 2:
            continue
        seeds.append(
            {
                "seed_id": f"code_{record['task_id']}",
                "domain": "code",
                "topic": infer_code_topic(record["text"]),
                "question": record["text"],
                "reference_code": record["code"],
                "test_setup_code": record.get("test_setup_code", ""),
                "test_list": record.get("test_list", []),
                "challenge_test_list": record.get("challenge_test_list", []),
                "source_dataset": "mbpp_train",
            }
        )
        code_count += 1
        if code_count >= NUM_CODE:
            break

    task_spec = {
        "project_goal": "Build a reproducible CoT + PRM mini dataset for math and code reasoning.",
        "seed_count": len(seeds),
        "domain_distribution": dict(Counter(seed["domain"] for seed in seeds)),
        "topic_distribution": dict(Counter(seed["topic"] for seed in seeds)),
        "trace_targets": {
            "positive": "correct reasoning trace",
            "negative": "corrupted or wrong reasoning trace",
            "repair": "wrong step followed by correction",
        },
        "validation_targets": [
            "final_answer_match",
            "step_level_quality_labels",
            "code_execution_and_unit_tests",
        ],
    }

    write_jsonl(seeds, SEED_FILE)
    write_json(task_spec, TASK_SPEC_FILE)
    print("✅ P6 种子池与任务规格生成完成。")
    print(task_spec)


if __name__ == "__main__":
    main()


### `src/generate_traces.py` - 生成推理轨迹

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter

from pipeline_utils import PROCESSED_DIR, corrupt_numeric_text, ensure_standard_dirs, load_jsonl, mutate_python_code, write_json, write_jsonl

SEED_FILE = PROCESSED_DIR / "seed_pool.jsonl"
TRACE_FILE = PROCESSED_DIR / "cot_traces.jsonl"
SUMMARY_FILE = PROCESSED_DIR / "trace_summary.json"


def build_math_traces(seed: dict) -> list[dict]:
    correct_steps = [
        {"step_idx": idx + 1, "text": step, "label": 1, "kind": "reasoning"}
        for idx, step in enumerate(seed["reference_steps"])
    ]
    final_step = {
        "step_idx": len(correct_steps) + 1,
        "text": f"Final answer: {seed['final_answer']}",
        "label": 1,
        "kind": "final",
    }
    positive = {
        "trace_id": f"{seed['seed_id']}_positive",
        "seed_id": seed["seed_id"],
        "domain": seed["domain"],
        "topic": seed["topic"],
        "trace_type": "positive",
        "question": seed["question"],
        "steps": correct_steps + [final_step],
        "candidate_answer": seed["final_answer"],
        "expected_answer": seed["final_answer"],
        "source_dataset": seed["source_dataset"],
    }

    wrong_steps = [dict(step) for step in correct_steps]
    wrong_steps[-1]["text"] = corrupt_numeric_text(wrong_steps[-1]["text"])
    wrong_steps[-1]["label"] = 0
    negative_final = {
        "step_idx": len(wrong_steps) + 1,
        "text": f"Final answer: {corrupt_numeric_text(seed['final_answer'])}",
        "label": 0,
        "kind": "final",
    }
    negative = {
        "trace_id": f"{seed['seed_id']}_negative",
        "seed_id": seed["seed_id"],
        "domain": seed["domain"],
        "topic": seed["topic"],
        "trace_type": "negative",
        "question": seed["question"],
        "steps": wrong_steps + [negative_final],
        "candidate_answer": negative_final["text"].split(":", 1)[1].strip(),
        "expected_answer": seed["final_answer"],
        "source_dataset": seed["source_dataset"],
    }

    repair_steps = wrong_steps + [negative_final]
    repair_steps.append(
        {
            "step_idx": len(repair_steps) + 1,
            "text": f"Correction: the previous arithmetic was wrong. The correct final answer is {seed['final_answer']}.",
            "label": 1,
            "kind": "repair",
        }
    )
    repair = {
        "trace_id": f"{seed['seed_id']}_repair",
        "seed_id": seed["seed_id"],
        "domain": seed["domain"],
        "topic": seed["topic"],
        "trace_type": "repair",
        "question": seed["question"],
        "steps": repair_steps,
        "candidate_answer": seed["final_answer"],
        "expected_answer": seed["final_answer"],
        "source_dataset": seed["source_dataset"],
    }
    return [positive, negative, repair]


def build_code_traces(seed: dict) -> list[dict]:
    tests = seed["test_list"] + seed.get("challenge_test_list", [])
    correct_steps = [
        {
            "step_idx": 1,
            "text": f"Understand the task: {seed['question']}",
            "label": 1,
            "kind": "analysis",
        },
        {
            "step_idx": 2,
            "text": f"Select a solution strategy for topic `{seed['topic']}` and prepare to validate with assertions.",
            "label": 1,
            "kind": "planning",
        },
        {
            "step_idx": 3,
            "text": seed["reference_code"],
            "label": 1,
            "kind": "code",
        },
        {
            "step_idx": 4,
            "text": f"Run {len(tests)} unit tests to confirm the implementation.",
            "label": 1,
            "kind": "verification",
        },
    ]
    positive = {
        "trace_id": f"{seed['seed_id']}_positive",
        "seed_id": seed["seed_id"],
        "domain": seed["domain"],
        "topic": seed["topic"],
        "trace_type": "positive",
        "question": seed["question"],
        "steps": correct_steps,
        "candidate_code": seed["reference_code"],
        "reference_code": seed["reference_code"],
        "test_setup_code": seed["test_setup_code"],
        "unit_tests": tests,
        "source_dataset": seed["source_dataset"],
    }

    broken_code = mutate_python_code(seed["reference_code"])
    negative_steps = [
        dict(correct_steps[0]),
        dict(correct_steps[1]),
        {
            "step_idx": 3,
            "text": broken_code,
            "label": 0,
            "kind": "code",
        },
        {
            "step_idx": 4,
            "text": "The implementation compiles conceptually, but the tests reveal a logic error.",
            "label": 0,
            "kind": "verification",
        },
    ]
    negative = {
        "trace_id": f"{seed['seed_id']}_negative",
        "seed_id": seed["seed_id"],
        "domain": seed["domain"],
        "topic": seed["topic"],
        "trace_type": "negative",
        "question": seed["question"],
        "steps": negative_steps,
        "candidate_code": broken_code,
        "reference_code": seed["reference_code"],
        "test_setup_code": seed["test_setup_code"],
        "unit_tests": tests,
        "source_dataset": seed["source_dataset"],
    }

    repair_steps = negative_steps + [
        {
            "step_idx": 5,
            "text": "Correction: revert the broken logic and restore the reference implementation before rerunning the tests.",
            "label": 1,
            "kind": "repair",
        }
    ]
    repair = {
        "trace_id": f"{seed['seed_id']}_repair",
        "seed_id": seed["seed_id"],
        "domain": seed["domain"],
        "topic": seed["topic"],
        "trace_type": "repair",
        "question": seed["question"],
        "steps": repair_steps,
        "candidate_code": seed["reference_code"],
        "reference_code": seed["reference_code"],
        "test_setup_code": seed["test_setup_code"],
        "unit_tests": tests,
        "source_dataset": seed["source_dataset"],
    }
    return [positive, negative, repair]


def main() -> None:
    ensure_standard_dirs()
    seeds = load_jsonl(SEED_FILE)
    traces: list[dict] = []

    for seed in seeds:
        if seed["domain"] == "math":
            traces.extend(build_math_traces(seed))
        else:
            traces.extend(build_code_traces(seed))

    summary = {
        "num_traces": len(traces),
        "trace_type_distribution": dict(Counter(trace["trace_type"] for trace in traces)),
        "domain_distribution": dict(Counter(trace["domain"] for trace in traces)),
        "step_label_distribution": dict(Counter(step["label"] for trace in traces for step in trace["steps"])),
    }

    write_jsonl(traces, TRACE_FILE)
    write_json(summary, SUMMARY_FILE)
    print("✅ P6 推理轨迹生成完成。")
    print(summary)


if __name__ == "__main__":
    main()


### `src/validate_and_score.py` - 校验并打分

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter

from pipeline_utils import PROCESSED_DIR, ensure_standard_dirs, load_jsonl, reward_bucket, run_python_snippet, write_json, write_jsonl

TRACE_FILE = PROCESSED_DIR / "cot_traces.jsonl"
VALIDATED_FILE = PROCESSED_DIR / "validated_traces.jsonl"
STEP_REWARD_FILE = PROCESSED_DIR / "step_rewards.jsonl"
SUMMARY_FILE = PROCESSED_DIR / "validation_summary.json"


def validate_math_trace(trace: dict) -> tuple[bool, dict]:
    expected = str(trace["expected_answer"]).strip()
    candidate = str(trace["candidate_answer"]).strip()
    passed = expected == candidate
    return passed, {
        "validation_mode": "final_answer_match",
        "expected_answer": expected,
        "candidate_answer": candidate,
        "stdout": candidate,
        "stderr": "",
    }


def validate_code_trace(trace: dict) -> tuple[bool, dict]:
    code_parts = [
        trace.get("test_setup_code", ""),
        trace["candidate_code"],
        "\n".join(trace["unit_tests"]),
    ]
    code = "\n\n".join(part for part in code_parts if part).strip() + "\n"
    passed, stdout, stderr = run_python_snippet(code)
    return passed, {
        "validation_mode": "execution_and_unit_tests",
        "expected_answer": "all_tests_pass",
        "candidate_answer": "all_tests_pass" if passed else "test_failure",
        "stdout": stdout,
        "stderr": stderr,
    }


def main() -> None:
    ensure_standard_dirs()
    traces = load_jsonl(TRACE_FILE)
    validated: list[dict] = []
    step_rewards: list[dict] = []

    for trace in traces:
        if trace["domain"] == "math":
            passed, validation = validate_math_trace(trace)
        else:
            passed, validation = validate_code_trace(trace)

        label_sum = sum(step["label"] for step in trace["steps"])
        score = label_sum / max(1, len(trace["steps"]))
        bucket = reward_bucket(score)

        enriched = dict(trace)
        enriched["validation"] = validation
        enriched["validation_passed"] = passed
        enriched["trace_score"] = round(score, 4)
        enriched["reward_bucket"] = bucket
        validated.append(enriched)

        for step in trace["steps"]:
            step_rewards.append(
                {
                    "trace_id": trace["trace_id"],
                    "seed_id": trace["seed_id"],
                    "domain": trace["domain"],
                    "trace_type": trace["trace_type"],
                    "step_idx": step["step_idx"],
                    "step_text": step["text"],
                    "step_kind": step["kind"],
                    "label": step["label"],
                    "trace_score": round(score, 4),
                    "reward_bucket": bucket,
                    "validation_passed": passed,
                }
            )

    summary = {
        "num_traces": len(validated),
        "validation_pass_rate": round(sum(item["validation_passed"] for item in validated) / max(1, len(validated)), 4),
        "trace_type_distribution": dict(Counter(item["trace_type"] for item in validated)),
        "domain_distribution": dict(Counter(item["domain"] for item in validated)),
        "reward_bucket_distribution": dict(Counter(item["reward_bucket"] for item in validated)),
    }

    write_jsonl(validated, VALIDATED_FILE)
    write_jsonl(step_rewards, STEP_REWARD_FILE)
    write_json(summary, SUMMARY_FILE)
    print("✅ P6 自动验证与打分完成。")
    print(summary)


if __name__ == "__main__":
    main()


### `src/prepare_prm_data.py` - 封装 PRM 数据

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict

from pipeline_utils import PROCESSED_DIR, TRAINING_DIR, deterministic_bucket, ensure_standard_dirs, estimated_tokens, load_jsonl, write_json, write_jsonl

INPUT_FILE = PROCESSED_DIR / "validated_traces.jsonl"
STEP_REWARD_FILE = PROCESSED_DIR / "step_rewards.jsonl"
FINAL_FILE = TRAINING_DIR / "prm_step_dataset.jsonl"
TRAIN_FILE = TRAINING_DIR / "train.jsonl"
VAL_FILE = TRAINING_DIR / "val.jsonl"
SMOKE_FILE = TRAINING_DIR / "smoke_test.jsonl"
MANIFEST_FILE = TRAINING_DIR / "training_manifest.json"


def build_step_record(trace: dict, step: dict) -> dict:
    prompt = trace["question"]
    previous_steps = [item["text"] for item in trace["steps"] if item["step_idx"] < step["step_idx"]]
    return {
        "record_id": f"{trace['trace_id']}_step_{step['step_idx']}",
        "trace_id": trace["trace_id"],
        "seed_id": trace["seed_id"],
        "domain": trace["domain"],
        "topic": trace["topic"],
        "trace_type": trace["trace_type"],
        "prompt": prompt,
        "previous_steps": previous_steps,
        "current_step": step["text"],
        "label": step["label"],
        "step_kind": step["kind"],
        "trace_score": trace["trace_score"],
        "reward_bucket": trace["reward_bucket"],
        "validation_passed": trace["validation_passed"],
    }


def main() -> None:
    ensure_standard_dirs()
    traces = load_jsonl(INPUT_FILE)
    step_records: list[dict] = []
    for trace in traces:
        for step in trace["steps"]:
            step_records.append(build_step_record(trace, step))

    step_records = sorted(step_records, key=lambda item: item["record_id"])
    train_records: list[dict] = []
    val_records: list[dict] = []
    smoke_by_domain: dict[str, list[dict]] = defaultdict(list)

    for record in step_records:
        if deterministic_bucket(record["trace_id"]) < 85:
            train_records.append(record)
        else:
            val_records.append(record)
        if len(smoke_by_domain[record["domain"]]) < 8:
            smoke_by_domain[record["domain"]].append(record)

    smoke_records: list[dict] = []
    for domain in sorted(smoke_by_domain):
        smoke_records.extend(smoke_by_domain[domain])

    manifest = {
        "num_records": len(step_records),
        "num_train_records": len(train_records),
        "num_val_records": len(val_records),
        "num_smoke_records": len(smoke_records),
        "domain_distribution": dict(Counter(record["domain"] for record in step_records)),
        "trace_type_distribution": dict(Counter(record["trace_type"] for record in step_records)),
        "label_distribution": dict(Counter(record["label"] for record in step_records)),
        "reward_bucket_distribution": dict(Counter(record["reward_bucket"] for record in step_records)),
        "estimated_tokens_total": sum(
            estimated_tokens(record["prompt"] + " " + " ".join(record["previous_steps"]) + " " + record["current_step"])
            for record in step_records
        ),
    }

    write_jsonl(step_records, FINAL_FILE)
    write_jsonl(train_records, TRAIN_FILE)
    write_jsonl(val_records, VAL_FILE)
    write_jsonl(smoke_records, SMOKE_FILE)
    write_json(manifest, MANIFEST_FILE)
    print("✅ P6 PRM 训练数据组织完成。")
    print(manifest)


if __name__ == "__main__":
    main()


### `src/evaluate_prm.py` - 评估 PRM 数据集

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter

from pipeline_utils import PROCESSED_DIR, REPORTS_DIR, TRAINING_DIR, ensure_standard_dirs, load_json, load_jsonl, write_json

METRICS_FILE = REPORTS_DIR / "p6_metrics.json"
REPORT_FILE = REPORTS_DIR / "p6_report.md"


def main() -> None:
    ensure_standard_dirs()
    seeds = load_jsonl(PROCESSED_DIR / "seed_pool.jsonl")
    traces = load_jsonl(PROCESSED_DIR / "validated_traces.jsonl")
    step_rewards = load_jsonl(PROCESSED_DIR / "step_rewards.jsonl")
    validation_summary = load_json(PROCESSED_DIR / "validation_summary.json")
    manifest = load_json(TRAINING_DIR / "training_manifest.json")

    positive_traces = [trace for trace in traces if trace["trace_type"] == "positive"]
    outcome_only_count = sum(trace["validation_passed"] for trace in traces)
    process_supervision_only_signal = sum(1 for step in step_rewards if step["label"] == 0)

    metrics = {
        "seed_count": len(seeds),
        "trace_count": len(traces),
        "step_count": len(step_rewards),
        "domain_distribution": dict(Counter(trace["domain"] for trace in traces)),
        "trace_type_distribution": dict(Counter(trace["trace_type"] for trace in traces)),
        "reward_bucket_distribution": dict(Counter(trace["reward_bucket"] for trace in traces)),
        "validation_pass_rate": validation_summary["validation_pass_rate"],
        "positive_trace_pass_rate": round(sum(trace["validation_passed"] for trace in positive_traces) / max(1, len(positive_traces)), 4),
        "outcome_only_supervision_count": outcome_only_count,
        "process_supervision_only_signal_steps": process_supervision_only_signal,
        "training_manifest": manifest,
        "estimated_manual_review_hours": round(len(traces) * 1.5 / 60, 2),
        "estimated_manual_review_cost_rmb": round(len(traces) * 1.5 * 100 / 60, 2),
        "estimated_external_generation_cost_usd": 0.0,
    }
    write_json(metrics, METRICS_FILE)

    report = f"""# P6 CoT and PRM Data Factory Report

## 1. 项目目标与任务设定

- 构建一个小型、可复现的 CoT 推理数据与 PRM 训练数据工厂。
- 任务覆盖：数学推理与代码推理两类任务。
- 监督目标：同时关注最终结果正确性与 step-level 过程质量。

## 2. 推理轨迹生成

- 种子数：{metrics['seed_count']}
- 轨迹总数：{metrics['trace_count']}
- 轨迹类型分布：{metrics['trace_type_distribution']}
- 领域分布：{metrics['domain_distribution']}

## 3. 自动验证与打分

- 验证通过率：{metrics['validation_pass_rate']}
- 正例轨迹通过率：{metrics['positive_trace_pass_rate']}
- 奖励桶分布：{metrics['reward_bucket_distribution']}
- 过程监督专有信号步数：{metrics['process_supervision_only_signal_steps']}

## 4. PRM 训练数据组织

- Step-level 样本数：{metrics['step_count']}
- 训练切分：train={manifest['num_train_records']} val={manifest['num_val_records']} smoke={manifest['num_smoke_records']}
- 标签分布：{manifest['label_distribution']}
- 奖励桶分布：{manifest['reward_bucket_distribution']}

## 5. 实验结果与复盘

- 仅结果监督可直接利用的轨迹数：{metrics['outcome_only_supervision_count']}
- 过程监督额外保留的错误步骤监督信号：{metrics['process_supervision_only_signal_steps']}
- 估算总 token 数：{manifest['estimated_tokens_total']}
- 人工抽检工时估算：{metrics['estimated_manual_review_hours']} 小时
- 人工抽检成本估算：{metrics['estimated_manual_review_cost_rmb']} 元

## 6. 扩展方向

- 扩展到科学推理、表格推理与代理规划任务。
- 引入更强的规则和执行器，做 finer-grained step reward 标注。
- 将 PRM 监督与在线采样闭环结合，支持更长轨迹的主动纠错。
"""

    REPORT_FILE.write_text(report, encoding="utf-8")
    print("✅ P6 报告生成完成。")
    print(report)


if __name__ == "__main__":
    main()


### `src/run_p6_checks.py` - 项目检查

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from collections import Counter
from datetime import UTC, datetime

from pipeline_utils import PROCESSED_DIR, REPORTS_DIR, ROOT_DIR, TRAINING_DIR, load_json, load_jsonl

SCRIPTS = sorted((ROOT_DIR / "src").glob("*.py"))
REQUIRED_FILES = [
    PROCESSED_DIR / "seed_pool.jsonl",
    PROCESSED_DIR / "task_spec.json",
    PROCESSED_DIR / "cot_traces.jsonl",
    PROCESSED_DIR / "trace_summary.json",
    PROCESSED_DIR / "validated_traces.jsonl",
    PROCESSED_DIR / "step_rewards.jsonl",
    PROCESSED_DIR / "validation_summary.json",
    TRAINING_DIR / "prm_step_dataset.jsonl",
    TRAINING_DIR / "train.jsonl",
    TRAINING_DIR / "val.jsonl",
    TRAINING_DIR / "smoke_test.jsonl",
    TRAINING_DIR / "training_manifest.json",
    REPORTS_DIR / "p6_metrics.json",
    REPORTS_DIR / "p6_report.md",
]
RESULTS_FILE = REPORTS_DIR / "p6_test_results.json"
REPORT_FILE = REPORTS_DIR / "p6_test_report.md"


def run_command(command: list[str]) -> dict:
    completed = subprocess.run(command, capture_output=True, text=True)
    return {
        "command": command,
        "returncode": completed.returncode,
        "passed": completed.returncode == 0,
        "stdout": completed.stdout.strip(),
        "stderr": completed.stderr.strip(),
    }


def main() -> None:
    command_checks = []
    py_compile = run_command([sys.executable, "-m", "py_compile", *[str(path) for path in SCRIPTS]])
    py_compile["name"] = "py_compile"
    command_checks.append(py_compile)

    evaluate = run_command([sys.executable, str(ROOT_DIR / "src" / "evaluate_prm.py")])
    evaluate["name"] = "evaluate_prm"
    command_checks.append(evaluate)

    traces = load_jsonl(PROCESSED_DIR / "validated_traces.jsonl")
    steps = load_jsonl(TRAINING_DIR / "prm_step_dataset.jsonl")
    smoke = load_jsonl(TRAINING_DIR / "smoke_test.jsonl")
    train = load_jsonl(TRAINING_DIR / "train.jsonl")
    val = load_jsonl(TRAINING_DIR / "val.jsonl")
    manifest = load_json(TRAINING_DIR / "training_manifest.json")

    dataset_checks = [
        {
            "name": "required_files_exist",
            "passed": all(path.exists() for path in REQUIRED_FILES),
            "details": {"missing_files": [str(path.relative_to(ROOT_DIR)) for path in REQUIRED_FILES if not path.exists()]},
        },
        {
            "name": "both_domains_present",
            "passed": {"math", "code"} <= {trace["domain"] for trace in traces},
            "details": {"domain_distribution": dict(Counter(trace["domain"] for trace in traces))},
        },
        {
            "name": "trace_types_present",
            "passed": {"positive", "negative", "repair"} <= {trace["trace_type"] for trace in traces},
            "details": {"trace_type_distribution": dict(Counter(trace["trace_type"] for trace in traces))},
        },
        {
            "name": "step_labels_cover_both_classes",
            "passed": {0, 1} <= {step["label"] for step in steps},
            "details": {"label_distribution": dict(Counter(step["label"] for step in steps))},
        },
        {
            "name": "reward_buckets_present",
            "passed": len({trace["reward_bucket"] for trace in traces}) >= 3,
            "details": {"reward_bucket_distribution": dict(Counter(trace["reward_bucket"] for trace in traces))},
        },
        {
            "name": "train_val_no_overlap",
            "passed": not ({record["record_id"] for record in train} & {record["record_id"] for record in val}),
            "details": {"overlap": len({record["record_id"] for record in train} & {record["record_id"] for record in val})},
        },
        {
            "name": "smoke_covers_both_domains",
            "passed": {"math", "code"} <= {record["domain"] for record in smoke},
            "details": {"smoke_domain_distribution": dict(Counter(record["domain"] for record in smoke))},
        },
        {
            "name": "manifest_matches_final_count",
            "passed": manifest["num_train_records"] + manifest["num_val_records"] == manifest["num_records"],
            "details": manifest,
        },
    ]

    results = {
        "timestamp_utc": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "overall_passed": all(item["passed"] for item in command_checks) and all(item["passed"] for item in dataset_checks),
        "total_checks": len(command_checks) + len(dataset_checks),
        "passed_checks": sum(item["passed"] for item in command_checks) + sum(item["passed"] for item in dataset_checks),
        "command_checks": command_checks,
        "dataset_checks": dataset_checks,
    }

    RESULTS_FILE.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    lines = [
        "# P6 Test Report",
        "",
        f"- Timestamp: {results['timestamp_utc']}",
        f"- Overall status: {'PASS' if results['overall_passed'] else 'FAIL'}",
        f"- Passed checks: {results['passed_checks']}/{results['total_checks']}",
        "",
        "## Command Checks",
        "",
    ]
    for check in command_checks:
        lines.append(f"- {check['name']}: {'PASS' if check['passed'] else 'FAIL'}")
        lines.append(f"  - Command: `{' '.join(check['command'])}`")
        if check["stdout"]:
            lines.append(f"  - Stdout: `{check['stdout'][:600]}`")
        if check["stderr"]:
            lines.append(f"  - Stderr: `{check['stderr'][:600]}`")

    lines.extend(["", "## Dataset Checks", ""])
    for check in dataset_checks:
        lines.append(f"- {check['name']}: {'PASS' if check['passed'] else 'FAIL'}")
        lines.append(f"  - Details: `{json.dumps(check['details'], ensure_ascii=False)}`")

    REPORT_FILE.write_text("\n".join(lines), encoding="utf-8")
    print(json.dumps(results, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


### `src/pipeline_utils.py` - 辅助脚本

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import hashlib
import json
import re
import subprocess
import sys
import tempfile
from pathlib import Path

ROOT_DIR = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
TRAINING_DIR = DATA_DIR / "training"
REPORTS_DIR = DATA_DIR / "reports"

P4_DATA_DIR = ROOT_DIR.parent / "project_4_synth" / "data"
GSM8K_FILE = P4_DATA_DIR / "gsm8k_train.jsonl"
MBPP_FILE = P4_DATA_DIR / "mbpp_train.jsonl"


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def ensure_standard_dirs() -> None:
    for path in [PROCESSED_DIR, TRAINING_DIR, REPORTS_DIR]:
        ensure_dir(path)


def write_json(data, path: Path) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_jsonl(records: list[dict], path: Path) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> list[dict]:
    records: list[dict] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def deterministic_bucket(key: str, buckets: int = 100) -> int:
    digest = hashlib.md5(key.encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % buckets


def estimated_tokens(text: str) -> int:
    compact = normalize_text(text)
    return max(1, len(compact) // 4)


def clean_gsm8k_answer(answer: str) -> str:
    answer = re.sub(r"<<([^>]+)>>", r"\1", answer)
    return answer.strip()


def extract_final_answer(answer: str) -> str | None:
    match = re.search(r"####\s*(.+)$", answer, re.MULTILINE)
    if match:
        return normalize_text(match.group(1))
    return None


def split_reasoning_steps(answer: str) -> list[str]:
    cleaned = clean_gsm8k_answer(answer)
    lines = [normalize_text(line) for line in cleaned.splitlines() if normalize_text(line)]
    steps = []
    for line in lines:
        if line.startswith("####"):
            continue
        steps.append(line)
    return steps


def first_number(text: str) -> str | None:
    match = re.search(r"-?\d[\d,]*(?:\.\d+)?", text)
    return match.group(0) if match else None


def corrupt_numeric_text(text: str) -> str:
    number = first_number(text)
    if not number:
        return text + " Therefore the final answer is 0."
    raw = number.replace(",", "")
    if "." in raw:
        new_value = str(round(float(raw) + 1.0, 2))
    else:
        new_value = str(int(raw) + 1)
    return text.replace(number, new_value, 1)


def mutate_python_code(code: str) -> str:
    for pattern, replacement in [
        (r"return\s+([^\n]+)", "return None"),
        (r"==", "!="),
        (r"\+", "-"),
    ]:
        mutated = re.sub(pattern, replacement, code, count=1)
        if mutated != code:
            return mutated
    return code + "\nraise AssertionError('broken trace')\n"


def run_python_snippet(code: str, timeout: int = 5) -> tuple[bool, str, str]:
    with tempfile.TemporaryDirectory() as tmp_dir:
        script = Path(tmp_dir) / "script.py"
        script.write_text(code, encoding="utf-8")
        completed = subprocess.run(
            [sys.executable, str(script)],
            capture_output=True,
            text=True,
            timeout=timeout,
        )
    return completed.returncode == 0, completed.stdout.strip(), completed.stderr.strip()


def reward_bucket(score: float) -> str:
    if score >= 0.95:
        return "high"
    if score >= 0.6:
        return "medium"
    if score > 0:
        return "low"
    return "fail"
